In [1]:
import numpy as np
import os
import cv2
import random
import shutil

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt
%matplotlib inline
import matplotlib.image as mpimg
import keras
from tensorflow.keras.models import Model, Sequential, load_model
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input, decode_predictions
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import img_to_array, load_img
from tensorflow.keras.layers import Dense, Conv2D, MaxPooling2D, Flatten, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam, RMSprop

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
width=300
height=300
channel=3

In [5]:
base_path=os.path.join(os.getcwd(), '/content/drive/MyDrive/인슐린')

In [6]:
import os

# 1. 상위 유형과 하위 라벨 매핑 정의
category_mapping = {
    '장시간형': ['란투스', '레버미어', '투제오', '트레시바'],
    '중간형': ['휴물린엔', '노보린엔'],
    '초속효형': ['노보래피드', '휴마로그', '애피드라', '피아스프'],
    '혼합형': ['노보믹스', '휴마로그믹스', '리조덱']
}

# 2. base_path 내의 실제 하위 폴더 목록 확인 및 계층적 분류
if os.path.exists(base_path):
    existing_subfolders = [f for f in os.listdir(base_path) if os.path.isdir(os.path.join(base_path, f))]

    print("--- 인슐린 계층 구조 분류 결과 ---")
    for parent, subs in category_mapping.items():
        found_subs = [sub for sub in subs if sub in existing_subfolders]
        if found_subs:
            print(f"[{parent}]")
            for sub in found_subs:
                sub_path = os.path.join(base_path, sub)
                num_images = len([img for img in os.listdir(sub_path) if img.lower().endswith(('.png', '.jpg', '.jpeg'))])
                print(f"  L {sub} (이미지 수: {num_images}개)")
        else:
            print(f"[{parent}] - 매칭되는 하위 폴더 없음")
else:
    print(f"경로를 찾을 수 없습니다: {base_path}")

--- 인슐린 계층 구조 분류 결과 ---
[장시간형] - 매칭되는 하위 폴더 없음
[중간형] - 매칭되는 하위 폴더 없음
[초속효형] - 매칭되는 하위 폴더 없음
[혼합형] - 매칭되는 하위 폴더 없음


In [12]:
import numpy as np
import tensorflow as tf
import os

# 1. 상위 유형과 하위 라벨 매핑 및 초기화
label_to_parent = {}
for parent, subs in category_mapping.items():
    for sub in subs:
        label_to_parent[sub] = parent

x_paths = []
y_sub_labels = []
y_parent_labels = []

# 2. 계층형 폴더 구조 탐색 (인슐린 / 상위유형 / 하위라벨 / 이미지 파일)
if os.path.exists(base_path):
    for parent_folder in os.listdir(base_path):
        parent_path = os.path.join(base_path, parent_folder)
        if os.path.isdir(parent_path):
            for sub_folder in os.listdir(parent_path):
                sub_path = os.path.join(parent_path, sub_folder)
                if os.path.isdir(sub_path):
                    # 이미지 파일 수집
                    for file_name in os.listdir(sub_path):
                        if file_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                            x_paths.append(os.path.join(sub_path, file_name))
                            y_sub_labels.append(sub_folder)
                            y_parent_labels.append(parent_folder)

# 3. 고유 라벨 목록 및 인덱스 매핑 구성
unique_sub_labels = sorted(list(set(y_sub_labels)))
unique_parent_labels = sorted(list(set(y_parent_labels)))

sub_label_to_idx = {label: i for i, label in enumerate(unique_sub_labels)}
parent_label_to_idx = {label: i for i, label in enumerate(unique_parent_labels)}

y_sub_idx = np.array([sub_label_to_idx[l] for l in y_sub_labels])
y_parent_idx = np.array([parent_label_to_idx[l] for l in y_parent_labels])

print(f"총 이미지 개수: {len(x_paths)}개")
print(f"분류 대상 상위 유형: {unique_parent_labels}")
print(f"분류 대상 하위 라벨: {unique_sub_labels}")

총 이미지 개수: 267개
분류 대상 상위 유형: ['장시간형', '중간형', '초속효형', '혼합형']
분류 대상 하위 라벨: ['노보래피드', '노보믹스30', '노보믹스50', '란투스', '레버미어', '애피드라', '투제오', '트레시바', '휴마로그', '휴마로그믹스25', '휴마로그믹스50', '휴물린 7030', '휴물린 엔']


In [13]:
# 4. 이미지 프리프로세싱 및 데이터 로더 정의
def load_and_preprocess_image(path, parent_label, sub_label):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, [height, width])
    image = image / 255.0
    return image, {'parent_output': parent_label, 'sub_output': sub_label}

# 5. 데이터 분할 및 TensorFlow Dataset API 구성
if len(x_paths) > 0:
    dataset = tf.data.Dataset.from_tensor_slices((x_paths, y_parent_idx, y_sub_idx))
    dataset = dataset.shuffle(buffer_size=len(x_paths))
    dataset = dataset.map(load_and_preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)

    # 데이터 증강 레이어 정의
    data_augmentation = tf.keras.Sequential([
        tf.keras.layers.RandomFlip("horizontal_and_vertical"),
        tf.keras.layers.RandomRotation(0.2),
        tf.keras.layers.RandomZoom(0.1)
    ])

    def augment_dataset(image, labels):
        return data_augmentation(image, training=True), labels

    train_size = int(0.8 * len(x_paths))
    train_ds = dataset.take(train_size).map(augment_dataset).batch(8).prefetch(tf.data.AUTOTUNE)
    val_ds = dataset.skip(train_size).batch(8).prefetch(tf.data.AUTOTUNE)
    print("데이터 파이프라인 구성 완료")
else:
    print("데이터셋이 존재하지 않습니다. 구글 드라이브 내 폴더 구조를 다시 확인해주세요.")

데이터 파이프라인 구성 완료


In [15]:
# 6. 전이 학습 기반 다중 출력 모델 정의 및 학습
if len(x_paths) > 0 and len(unique_parent_labels) > 0:
    base_model = tf.keras.applications.MobileNetV2(input_shape=(height, width, channel), include_top=False, weights='imagenet')
    base_model.trainable = False

    inputs = tf.keras.layers.Input(shape=(height, width, channel))
    x = base_model(inputs, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dense(128, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.3)(x)

    # 상위 유형 및 하위 라벨 예측 헤드 생성
    parent_output = tf.keras.layers.Dense(len(unique_parent_labels), activation='softmax', name='parent_output')(x)
    sub_output = tf.keras.layers.Dense(len(unique_sub_labels), activation='softmax', name='sub_output')(x)

    model = tf.keras.models.Model(inputs=inputs, outputs=[parent_output, sub_output])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss={
            'parent_output': 'sparse_categorical_crossentropy',
            'sub_output': 'sparse_categorical_crossentropy'
        },
        loss_weights={'parent_output': 1.0, 'sub_output': 1.0},
        metrics={
            'parent_output': 'accuracy',
            'sub_output': 'accuracy'
        }
    )

    model.fit(train_ds, validation_data=val_ds, epochs=15)
else:
    print("모델 빌드가 취소되었습니다. 학습용 데이터가 준비되지 않았습니다.")

/tmp/ipykernel_695/2037226245.py:3: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = tf.keras.applications.MobileNetV2(input_shape=(height, width, channel), include_top=False, weights='imagenet')


Epoch 1/15
27/27 ━━━━━━━━━━━━━━━━━━━━ 154s 5s/step - loss: 3.8286 - parent_output_accuracy: 0.4554 - parent_output_loss: 1.2771 - sub_output_accuracy: 0.1502 - sub_output_loss: 2.5390 - val_loss: 3.1929 - val_parent_output_accuracy: 0.5741 - val_parent_output_loss: 1.0007 - val_sub_output_accuracy: 0.2778 - val_sub_output_loss: 2.1877
Epoch 2/15
27/27 ━━━━━━━━━━━━━━━━━━━━ 16s 606ms/step - loss: 3.1676 - parent_output_accuracy: 0.5446 - parent_output_loss: 1.0131 - sub_output_accuracy: 0.2582 - sub_output_loss: 2.1475 - val_loss: 2.8146 - val_parent_output_accuracy: 0.5926 - val_parent_output_loss: 0.8307 - val_sub_output_accuracy: 0.2778 - val_sub_output_loss: 1.9723
Epoch 3/15
27/27 ━━━━━━━━━━━━━━━━━━━━ 17s 619ms/step - loss: 2.7643 - parent_output_accuracy: 0.6338 - parent_output_loss: 0.8999 - sub_output_accuracy: 0.3709 - sub_output_loss: 1.8662 - val_loss: 2.2195 - val_parent_output_accuracy: 0.6852 - val_parent_output_loss: 0.7501 - val_sub_output_accuracy: 0.6667 - val_sub_outpu

In [16]:
# 7. TFLite 파일로 변환 및 저장
try:
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    tflite_model = converter.convert()

    tflite_path = os.path.join(base_path, 'insulin_classifier.tflite')
    with open(tflite_path, 'wb') as f:
        f.write(tflite_model)

    print(f"TFLite 모델 저장 완료: {tflite_path}")
except NameError:
    print("모델 객체가 정의되어 있지 않습니다. 학습 코드 블록이 성공적으로 실행되었는지 확인해주세요.")

Saved artifact at '/tmp/tmpm1s1s3qz'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 300, 300, 3), dtype=tf.float32, name='keras_tensor_478')
Output Type:
  List[TensorSpec(shape=(None, 4), dtype=tf.float32, name=None), TensorSpec(shape=(None, 13), dtype=tf.float32, name=None)]
Captures:
  136403332274256: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136403329999952: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136403329999760: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136403330000144: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136403329998992: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136403329997072: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136403330000912: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136403330000720: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136403330001104: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136403329999376

In [11]:
# 8. 새로운 임의의 이미지 예측을 위한 함수 정의
def predict_new_image(img_path):
    img = tf.io.read_file(img_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [height, width])
    img = tf.expand_dims(img / 255.0, axis=0)

    parent_preds, sub_preds = model.predict(img)

    parent_idx = np.argmax(parent_preds[0])
    sub_idx = np.argmax(sub_preds[0])

    predicted_parent = unique_parent_labels[parent_idx]
    predicted_sub = unique_sub_labels[sub_idx]

    print(f"분류 결과: 상위유형 [{predicted_parent}] -> 하위라벨 [{predicted_sub}]")